# 5. Prepare model features

The main workflow is **04 windows → 05 features → 06 training**. `analysis_mode = "pilot"` permits unmet collection budgets; `"strict"` requires those budgets. Both modes keep the same timing, completeness and parcel-separation checks. No imagery is downloaded here. Previous archives, splits, models and immutable bundles remain preserved.


## 1. Setup and configuration

Read notebook 04’s explicit handoff. The main pilot/strict path does not require production freezes. Changing the feature policy creates a separate bundle.


In [ ]:
from pathlib import Path
import json
import os
import sys
repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
os.chdir(repository_root)
from crop_pipeline import analysis_workflow, progress as pipeline_progress
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
from crop_pipeline.sources import adaptive, annual, pilot_sources
from crop_pipeline.windows import seasons as season_workflow
from crop_pipeline.common import save_json
from IPython.display import display

analysis_mode = "pilot"  # pilot: report unmet budgets; strict: require collection targets
analysis_workflow.validate_mode(analysis_mode)
local_settings = json.loads(Path("settings.local.json").read_text()) if Path("settings.local.json").exists() else {}
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
source_run_name = local_settings.get("analysis_collection", adaptive.COLLECTION_NAME)
source_manifest_paths, analysis_run_name = pilot_sources.resolve_collection(report_folder, source_run_name)
analysis_folder = report_folder.parent / "analysis" / analysis_run_name
season_folder = analysis_folder / "seasons"
workflow_folder = season_folder / "workflow" / analysis_mode
print("Collection:", source_run_name, "| Mode:", analysis_mode)
print("Workflow folder:", workflow_folder)

from crop_pipeline.features import experiments as feature_experiments
feature_policy = dict(feature_experiments.FEATURE_POLICY)
export_formal_handoffs = False  # Optional reviewed/frozen production branch
feature_policy_frozen = False
feature_development_evidence = ""
stages = ["Load window handoff", "Check window inputs", "Calculate feature designs", "Save feature handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("05_feature_engineering", stages, report_folder.parent / "progress", {"analysis": analysis_run_name, "mode": analysis_mode})
analysis_workflow.write_stage_pointer(workflow_folder / "feature_stage.json", status="pending", reason="Notebook 05 has not finished the current feature export")


## 2. Load selected windows

Verify hashes and the current stage pointer. Missing or pending inputs stop this notebook; it never substitutes an older analysis.


In [ ]:
pipeline_progress.begin_stage("Load window handoff")
window_handoff = analysis_workflow.resolve_stage_pointer(workflow_folder / "window_stage.json")
window_rows, configuration, coverage, window_manifest = analysis_workflow.load_window_handoff(window_handoff)
if configuration["analysis_mode"] != analysis_mode:
    raise ValueError("Window mode differs from the requested feature mode")
analysis_workflow.require_collection_readiness(report_folder, analysis_mode)
print("Window handoff:", window_handoff)
print("Usable pixel windows:", len(window_rows))
pipeline_progress.end_stage()


## 3. Check identities and partitions

Keep the same sample rows, window dates, crop labels and locked parcel partitions for all designs. Missing values and invalid identities remain errors.


In [ ]:
pipeline_progress.begin_stage("Check window inputs")
feature_experiments.validate_policy(feature_policy)
from crop_pipeline.training.exploratory import split_summary
show_table(split_summary(window_rows), title="Feature input samples by crop and partition")
print("Collection targets met:", coverage["targets_met"])
print("Production settings frozen:", configuration["frozen"])
pipeline_progress.end_stage()


## 4. Calculate the three feature designs

| Design | Features |
| --- | ---: |
| Interval measurements | 72 |
| Temporal shape | 48 |
| Combined | 120 |

Add SAVI with `L = 0.5` to the five bands and NDVI/NDRE/NDMI. Shape features use actual calendar-bin midpoints. Undefined indices are errors; no samples are silently dropped.


In [ ]:
pipeline_progress.begin_stage("Calculate feature designs")
# Calculate once here. Notebook 06 consumes the saved values without recomputing them.
feature_handoff = feature_experiments.export_pilot_feature_handoff(
    window_handoff, workflow_folder / "features", policy=feature_policy,
)
feature_rows, configuration, coverage, feature_manifest = feature_experiments.load_pilot_feature_handoff(feature_handoff)
for design, columns in feature_experiments.feature_sets().items():
    print(design, ":", len(columns), "predictors")
pipeline_progress.end_stage()


## 5. Save the feature handoff

Save enhanced rows, predictor order, feature policy, coverage and window provenance in a verified bundle. Notebook 06 reads this bundle directly. Advanced formal exports remain optional and retain production checks.


In [ ]:
pipeline_progress.begin_stage("Save feature handoff")
analysis_workflow.write_stage_pointer(workflow_folder / "feature_stage.json", feature_handoff)
feature_handoffs = {"pilot": feature_handoff}
if export_formal_handoffs:
    for window_folder, feature_folder in [("training_input", "feature_input"), ("exploratory_training_input", "exploratory_feature_input")]:
        formal_path = feature_experiments.export_feature_handoff(
            season_folder / window_folder / "training_handoff.json", season_folder / feature_folder,
            policy=feature_policy, frozen=feature_policy_frozen,
            development_evidence=feature_development_evidence,
        )
        feature_handoffs[feature_folder] = formal_path
        print("Advanced formal export:", formal_path)
print("Feature handoff:", feature_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

Confirm row counts, unchanged partitions and the saved path. Continue to 06; do not select features using Paoay results. See [feature designs](../docs/feature_experiments.md).


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Status: prepared for local", analysis_mode, "training")
print("Pixel windows:", feature_manifest["row_count"], "| Partitions: unchanged")
show_mapping(feature_policy, title="Feature recipe", key_label="Setting", humanize_values=True)
print("Saved bundle:", feature_handoff.parent)
print("Next: run 06_model_training.ipynb with the same analysis_mode.")
print("Formal production readiness is separate; no review or freeze was marked complete.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
